# PARTE C (Experimento Avanzado): Regresión Directa con Deep Learning (Transformers - BETO)

## 1. Introducción al Experimento
Tras desarrollar un modelo altamente interpretable mediante ABSA y Random Forest (Parte B), nos planteamos un reto técnico adicional: **¿Qué ocurriría si eliminamos la extracción de tópicos y le pasamos el texto libre directamente a una red neuronal profunda?**

Para ello, hemos diseñado un experimento de **Regresión Directa con Transformers**. En lugar de usar índices numéricos, alimentamos a un modelo de lenguaje masivo (LLM) con el texto original redactado por el cliente, pidiéndole que aprenda los patrones lingüísticos complejos que derivan en una nota concreta.

## 2. Metodología
* **El Modelo:** Hemos utilizado **BETO** (`dccuchile/bert-base-spanish-wwm-uncased`), un modelo de la arquitectura BERT preentrenado exclusivamente con corpus en español, ideal para captar la semántica, ironía y contexto de las reseñas de Booking.
* **Adaptación del Dataset:** Concatenamos las columnas de comentarios positivos y negativos en un único bloque de texto continuo por cliente (Ej: *"Lo bueno: La cama. Lo malo: El ruido"*).
* **La Tarea (Sequence Classification for Regression):** Modificamos la capa de salida del modelo Transformer para que devuelva un único valor continuo (la nota predicha) en lugar de una categoría discreta, optimizando mediante el Error Cuadrático Medio (MSE).

## 3. Objetivo de Negocio: Trade-off Técnico
El objetivo de este experimento no es sustituir a nuestro modelo de Random Forest, sino ilustrar el clásico dilema en Data Science: **Interpretabilidad vs. Precisión (Caja Blanca vs. Caja Negra)**.
Mientras que el Transformer es capaz de procesar la semántica pura del texto, su naturaleza de "caja negra" nos impide generar el gráfico de *Feature Importance* vital para las recomendaciones de negocio.

In [1]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [8]:
# ==============================================================================
# PARTE C: DEEP LEARNING — REGRESIÓN CON TRANSFORMERS (BETO)
# ==============================================================================

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

use_gpu = torch.cuda.is_available()
print(f"¿GPU detectada?: {'✅ SÍ' if use_gpu else '❌ NO (¡Cuidado, irá muy lento!)'}")

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# 1. PREPARACIÓN DEL TEXTO (Train / Val / Test)
# ------------------------------------------------------------------------------
print("\n⏳ Preparando los textos para el Transformer...")
df = pd.read_csv('df_comentarios_final.csv')

df = df.dropna(subset=['nota']).copy()
df['positivo'] = df['positivo'].fillna('Nada')
df['negativo'] = df['negativo'].fillna('Nada')

df['texto_completo'] = "Lo positivo: " + df['positivo'] + " . Lo negativo: " + df['negativo']

# ¡AQUÍ DECIDES! Si quieres usar todas, comenta esta línea. Si tarda mucho, déjala en 20000.
# df = df.sample(20000, random_state=42)

df_dl = df[['texto_completo', 'nota']].rename(columns={'texto_completo': 'text', 'nota': 'label'})
df_dl['label'] = df_dl['label'].astype('float32')

# --- EL REPARTO (80% Train, 10% Val, 10% Test) ---
# Primero partimos: 80% Train y 20% Temporal
train_df, temp_df = train_test_split(df_dl, test_size=0.2, random_state=42)
# Luego partimos ese 20% por la mitad: 10% Validation y 10% Test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"📚 Tamaños -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

dataset_train = Dataset.from_pandas(train_df, preserve_index=False)
dataset_val   = Dataset.from_pandas(val_df, preserve_index=False)
dataset_test  = Dataset.from_pandas(test_df,  preserve_index=False)

# ------------------------------------------------------------------------------
# 2. TOKENIZACIÓN
# ------------------------------------------------------------------------------
print("🧠 Descargando Tokenizador BETO (Spanish BERT)...")
modelo_nombre = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)

def tokenizar_funcion(ejemplos):
    return tokenizer(ejemplos["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = dataset_train.map(tokenizar_funcion, batched=True)
tokenized_val   = dataset_val.map(tokenizar_funcion, batched=True)
tokenized_test  = dataset_test.map(tokenizar_funcion,  batched=True)

# ------------------------------------------------------------------------------
# 3. CONFIGURACIÓN DEL MODELO
# ------------------------------------------------------------------------------
print("🤖 Descargando Modelo BETO adaptado para Regresión...")
# num_labels=1 → regresión (predice un número continuo, no una categoría)
model = AutoModelForSequenceClassification.from_pretrained(modelo_nombre, num_labels=1)

metric_mse = evaluate.load("mse")
metric_mae = evaluate.load("mae")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # BETO devuelve shape (N, 1) en regresión → hay que aplanar a (N,)
    predictions = predictions.squeeze()
    mse = metric_mse.compute(predictions=predictions, references=labels)["mse"]
    mae = metric_mae.compute(predictions=predictions, references=labels)["mae"]
    return {"rmse": round(float(np.sqrt(mse)), 4), "mae": round(float(mae), 4)}

training_args = TrainingArguments(
    output_dir="./beto_hotel_reviews",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",        # muestra métricas una vez por época, no cada paso
    report_to="none",                # evita que intente conectar con wandb/tensorboard
    fp16=use_gpu,                    # fp16 solo si hay GPU; en CPU causa crash
    dataloader_num_workers=0,        # evita warnings de multiprocesing en Colab
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# ------------------------------------------------------------------------------
# 4. ENTRENAMIENTO Y EVALUACIÓN FINAL
# ------------------------------------------------------------------------------
print("\n🚀 ¡INICIANDO ENTRENAMIENTO DE DEEP LEARNING!")
trainer.train()

print("\n📊 EXAMEN FINAL: EVALUANDO EL MODELO EN EL CONJUNTO DE TEST PURO...")
# Ojo, usamos predict() pasándole explícitamente el test
resultados_test = trainer.predict(tokenized_test)
metricas_test = resultados_test.metrics

print("\n" + "="*55)
print("🏆 RESULTADOS FINALES DEL TRANSFORMER (BETO) EN TEST")
print("="*55)
# HuggingFace le pone el prefijo 'test_' cuando usas predict()
print(f"  MAE  (Error Absoluto Medio)  : {metricas_test['test_mae']:.2f} puntos")
print(f"  RMSE (Raíz Error Cuadrático) : {metricas_test['test_rmse']:.2f}")
print("="*55)

¿GPU detectada?: ✅ SÍ

⏳ Preparando los textos para el Transformer...
📚 Tamaños -> Train: 46976 | Val: 5872 | Test: 5872
🧠 Descargando Tokenizador BETO (Spanish BERT)...


Map:   0%|          | 0/46976 [00:00<?, ? examples/s]

Map:   0%|          | 0/5872 [00:00<?, ? examples/s]

Map:   0%|          | 0/5872 [00:00<?, ? examples/s]

🤖 Descargando Modelo BETO adaptado para Regresión...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


📊 EXAMEN FINAL: EVALUANDO EL MODELO EN EL CONJUNTO DE TEST PURO...



🏆 RESULTADOS FINALES DEL TRANSFORMER (BETO) EN TEST
  MAE  (Error Absoluto Medio)  : 0.95 puntos
  RMSE (Raíz Error Cuadrático) : 1.31


In [9]:
# ==============================================================================
# 5. EJEMPLO VISUAL: PREDICCIONES REALES DEL TRANSFORMER
# ==============================================================================
import torch

print("\n🔍 EJEMPLO VISUAL: ¿Qué está prediciendo BETO exactamente?")

# Tomamos 5 reseñas aleatorias del conjunto de test que el modelo nunca ha visto en el entrenamiento
ejemplos_test = test_df.sample(5, random_state=10) # Cambia el número si quieres ver ejemplos distintos

textos = ejemplos_test['text'].tolist()
notas_reales = ejemplos_test['label'].tolist()

# Tokenizamos los textos de ejemplo (los preparamos para BETO)
inputs = tokenizer(textos, padding=True, truncation=True, max_length=128, return_tensors="pt")

# Movemos los datos a la GPU (si la estamos usando) para que sea instantáneo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Hacemos la predicción (model.eval() apaga el modo entrenamiento)
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    # BETO devuelve tensores matemáticos, los aplanamos y los pasamos a números normales (numpy)
    predicciones = outputs.logits.squeeze().cpu().numpy()

# Creamos un DataFrame para visualizarlo de forma elegante
df_ejemplos = pd.DataFrame({
    # Recortamos el texto a 100 caracteres para que la tabla no sea kilométrica en pantalla
    'Texto Analizado (Input BETO)': [t[:100] + '...' if len(t) > 100 else t for t in textos],
    'Nota Real (Cliente)': notas_reales,
    'Nota Predicha (BETO)': np.round(predicciones, 1)
})

# Calculamos el error exacto en cada ejemplo
df_ejemplos['Error Absoluto'] = np.abs(df_ejemplos['Nota Real (Cliente)'] - df_ejemplos['Nota Predicha (BETO)'])

display(df_ejemplos)

print("\n💡 Observación:")
print("Fíjate en el texto de los que tienen menor 'Error Absoluto'. El Transformer es capaz")
print("de leer palabras como 'horrible', 'perfecto', o el sarcasmo, y ajusta la nota matemática")
print("sin que nosotros hayamos tenido que extraer los tópicos previamente.")


🔍 EJEMPLO VISUAL: ¿Qué está prediciendo BETO exactamente?


,Texto Analizado (Input BETO),Nota Real (Cliente),Nota Predicha (BETO),Error Absoluto
0,Lo positivo: Maravillosa vista y posición estr...,8.0,8.8,0.8
1,Lo positivo: ra ajky En Borné . Lo negativo: R...,9.0,8.0,1.0
2,Lo positivo: excelente ubicación céntrica. Hab...,10.0,9.7,0.3
3,Lo positivo: ¡Tuvimos una estancia increíble e...,10.0,9.9,0.1
4,Lo positivo: Nada . Lo negativo: Nada,9.0,8.8,0.2



💡 Observación:
Fíjate en el texto de los que tienen menor 'Error Absoluto'. El Transformer es capaz
de leer palabras como 'horrible', 'perfecto', o el sarcasmo, y ajusta la nota matemática
sin que nosotros hayamos tenido que extraer los tópicos previamente.
